# 14. Monte Carlo pilot on the PSMA phantoms

Dose of a voxel in the virtual cohort is linear in the time-integrated activity (TIA). If the dose per unit TIA of each compartment, $D_c(\mathbf{x})$, is known, any PBPK draw gives
$$D(\mathbf{x}) = \sum_c \mathrm{TIA}_c\, D_c(\mathbf{x})$$
without rerunning GATE. This notebook tests that on 3 phantoms of notebook 11 and asks how much Monte Carlo the training split needs.

**Steps**
1. Phantoms: one typical, one with high TLV, one with a single kidney.
2. Material map: Schneider 2000 from the CT, ICRP 110 densities in the contrast-affected organs.
3. Sources: unit TIA per compartment, PET-weighted and uniform.
4. Dose matrix $D_c$ per target.
5. Linearity: $\sum \mathrm{TIA}_c D_c$ against a direct run with a combined source.
6. VSV convolution with density correction against Monte Carlo.
7. Organ S-values, self and cross.
8. PET-weighted against uniform source.
9. Statistical uncertainty and runtime against the number of primaries.
10. Cost for the training split and proposal.

GATE runs in `scripts/monte_carlo/run_mc_pilot.py` (resumable, one process per batch, 4 threads); it calls `scripts/gate/lu177_patient_mc.py`, validated in notebook 05, with the same physics. This notebook reads its results, and the first code cell after section 3 runs it again only if something is missing. Code: `src/psma_mc.py`. Outputs: `data/processed/psma/mc_pilot/`.

**About this notebook.** GATE Monte Carlo pilot on 3 phantoms: dose matrix, linearity, VSV and S-value comparisons, cost.
- **Inputs:** phantoms; `scripts/monte_carlo/run_mc_pilot.py`, `scripts/dataset/fetch_opendose_svalues.py`.
- **Outputs:** `data/processed/psma/mc_pilot/`.
- **Run time:** about 2 min with the runs done (the GATE runs are separate).
- **Environment:** .venv (opengate).

In [ ]:
import json, sys, time
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd

WORKSPACE = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "paths.py").exists())
sys.path.insert(0, str(WORKSPACE / "src"))
sys.path.insert(0, str(WORKSPACE / "scripts" / "monte_carlo"))
import psma_mc as M
import run_mc_pilot as R

pd.set_option("display.width", 220, "display.max_columns", 40, "display.max_rows", 200, "display.float_format", lambda v: f"{v:.3g}")
plt.rcParams.update({"figure.dpi": 90, "font.size": 9, "axes.spines.top": False, "axes.spines.right": False})
INK, GRID = "#52514e", "#e5e4e0"
STUDY_COLOR = dict(zip(R.STUDIES, ["#2a78d6", "#eb6834", "#1baf7a"]))
STUDY_NAME = dict(zip(R.STUDIES, ["typical", "high TLV", "single kidney"]))
COMPARTMENTS = list(M.COMPARTMENTS)
TARGETS = list(M.TARGETS)
SELF_TARGET = {c: c for c in COMPARTMENTS}
DATA = M.PILOT
QC = DATA / "figures"
QC.mkdir(exist_ok=True)
available = [s for s in R.STUDIES if (DATA / s).exists()]
targets_raw = pd.read_csv(DATA / "targets.csv")
print(targets_raw.groupby(["study_id", "variant"]).size())

## 1. Phantoms

All from the 68Ga training split of notebook 08 (180 studies, 132 subjects).

- **Typical:** the unflagged study closest to the medians of the split on weight, kidney, liver and spleen volume, TLV and uptake time (largest absolute standardised log distance, smallest over the 58 studies without QC flags).
- **High TLV:** the highest TLV of the split, a stress test for the grouped lesion source.
- **Single kidney:** one of the 2 studies of the split with `single_kidney`, the one with no second kidney in the segmentation at all.

In [ ]:
summary = pd.read_csv(M.ph.psma_data.PROCESSED / "phantoms_summary.csv")
split = pd.read_csv(M.ph.psma_data.PROCESSED / "psma_68ga_split.csv")
train = summary.merge(split[["study_id", "split"]], on="study_id").query("split == 'train' and pbpk_cohort")
flags = [c for c in train.columns if c.startswith("flag__") and c not in ("flag__nb09_flags", "flag__contrast_metadata_mismatch", "flag__salivary_absent")]
clean = train[np.logical_and.reduce([train[c].astype(object).where(train[c].notna(), "").astype(str).isin(["", "False"]) for c in flags])]
features = ["weight_kg", "kidneys_total__volume_ml", "liver__volume_ml", "spleen__volume_ml", "lesions_volume_ml", "uptake_time_min"]
logs = np.log(train[features].clip(lower=0.1))
distance = ((np.log(clean[features].clip(lower=0.1)) - logs.median()) / logs.std()).abs().max(axis=1)
assert clean.loc[distance.idxmin(), "study_id"] == list(R.STUDIES)[0]
assert train.sort_values("lesions_volume_ml").iloc[-1]["study_id"] == list(R.STUDIES)[1]
assert list(R.STUDIES)[2] in set(train.loc[train["flag__kidney_absent_in_segmentation"].astype(object).where(train["flag__kidney_absent_in_segmentation"].notna(), "").astype(str).str.len() > 0, "study_id"])
print(len(train), "training studies,", len(clean), "without flags")
grid_of = lambda s: json.loads((M.ph.PHANTOMS / s / "meta.json").read_text())["grid"]
print(summary.set_index("study_id").loc[list(R.STUDIES), ["scanner", "weight_kg", "uptake_time_min"] + features[1:-1] + ["kidney_functional__volume_ml", "n_lesions", "lesions_total_pct_ia", "totals__fov_pct_ia"]].T.to_string())
print({STUDY_NAME[s]: (grid_of(s)["shape"], [round(z, 2) for z in grid_of(s)["spacing_mm"]]) for s in R.STUDIES})

## 2. Material map

Outside the override, materials and densities are the Schneider 2000 table of notebook 05, with density tolerance 0.05 g/cm³. Contrast raises the HU of kidneys, liver, spleen and vessels and the urine HU, which Schneider would read as denser tissue, in places as bone-marrow mixtures. So these compartments get the ICRP 110 Table A.1 adult male densities of notebook 11: kidney 1.05 (fluid voxels 1.04), liver 1.05, spleen 1.04, salivary glands 1.03, bladder 1.04, blood pool 1.06 g/cm³.

**How.** Each overridden compartment gets a value outside the HU range in a copy of the CT (10000 and up) and its own material, through the option `--material-overrides` added to `lu177_patient_mc.py`. Default behaviour and the runs of notebook 05 do not change. The density is exact, not binned. **Composition is an assumption:** the Schneider soft-tissue row (HU 19-79) for all of them, not an ICRP organ composition. At these energies composition matters little for soft tissue; this was not tested.

**Not overridden, as requested:** GI tract (oral contrast and gas stay as in the CT), lesions (Schneider from the possibly enhanced CT) and the vessels other than the eroded aorta.

The table gives what Schneider would have assigned: mean density and share of voxels above 1.1 g/cm³.

In [ ]:
rows = []
for study in R.STUDIES:
    ct, organs = M.load(study, "ct"), M.load(study, "organs")
    schneider = M.ph.density_from_hu(ct)
    for label, name in M.OVERRIDE_LABELS.items():
        mask = organs == label
        if mask.sum() < 10:
            continue
        rows.append(dict(phantom=STUDY_NAME[study], compartment=name, voxels=int(mask.sum()), mean_hu=ct[mask].mean(), schneider_g_ml=schneider[mask].mean(), share_above_1p1=(schneider[mask] > 1.1).mean(), icrp110_g_ml=M.ph.NOMINAL_DENSITY[name]))
materials = pd.DataFrame(rows)
materials["icrp_over_schneider"] = materials["icrp110_g_ml"] / materials["schneider_g_ml"]
materials.to_csv(DATA / "material_override.csv", index=False)
print(materials.round(3).to_string(index=False))

**Reading.** Schneider would give kidneys, liver, spleen and blood pool about 1.09-1.10 g/mL, and put 23-64% of kidney and 31-37% of liver and spleen voxels above 1.1 g/mL, into bone-marrow mixtures. Since dose goes as 1/mass, this alone is a 3-5% effect on these organs (`icrp_over_schneider` 0.95-0.97). Bladder and kidney fluid move by 1-2%. The parotids go the other way in two phantoms (mean HU down to -48, ratio up to 1.06): they are fatty, and the fixed density may then be too high.

## 3. Sources

For each compartment $c$ a source with unit total, weights $w_c(\mathbf{x})$:
- **PET:** the real 68Ga PET inside the compartment;
- **uniform:** constant per unit volume inside it, as a control.

| Compartment | Labels of `organs.nii.gz` |
|---|---|
| `kidneys_functional` | 1 |
| `liver`, `spleen`, `bladder`, `gi_tract` | 3, 4, 9, 10 |
| `salivary_glands` | 5-8, all four, truncated parotids included |
| `lesions` | 15, grouped |
| `rest_of_body` | 14 + blood pool 11 + spine 12, 13 + kidney fluid 2 |

**Choice:** the source list has no separate blood or marrow source, so blood pool, spine and kidney fluid join the rest of body, and activity is conserved in the body mask. They stay as *targets*, so marrow dose can be read. A later version can split them if the PBPK needs marrow self-dose. Activity outside the body mask (median 1.7 %IA, notebook 11) has no source.

**PET spill-out goes with the source.** The PET is blurred and misregistered: notebook 11 found that the kidneys lose about 30% of their activity from the CT mask. That activity ends in the PET-weighted `rest_of_body`, so its source has a halo around kidneys and bladder.

Dose per unit TIA from a run of $N$ decays: $D_c = \text{dose} / N \times 3.6\times10^{9}$ decays per MBq·h, in Gy/(MBq·h). PET-weighted: 5 batches of 2×10⁵ decays on the typical phantom, 3 on the others. Uniform: one batch. Physics as in notebook 05: `G4EmStandardPhysics_option4`, cut 1 mm, full Lu-177 decay, dose scored on the PET grid.

In [ ]:
missing = [s for s in R.STUDIES if not (DATA / s / "D_combined_direct_gy.nii.gz").exists() and s == R.COMBINED_STUDY]
print("Missing combined run:", missing, "| to rerun: python scripts/monte_carlo/run_mc_pilot.py")
rows = []
for study in available:
    pet, organs = M.load(study, "pet_pct_ia_ml"), M.load(study, "organs")
    voxel_ml = np.prod(M.spacing_of(study)) / 1000
    for c, labels in M.COMPARTMENTS.items():
        mask = np.isin(organs, labels)
        w = pet[mask].astype(np.float64)
        rows.append(dict(phantom=STUDY_NAME[study], compartment=c, volume_ml=mask.sum() * voxel_ml, pct_ia=w.sum() * voxel_ml,
                         effective_volume_ml=(w.sum() ** 2 / (w ** 2).sum()) * voxel_ml if w.sum() else np.nan))
sources = pd.DataFrame(rows)
sources["effective_over_volume"] = sources["effective_volume_ml"] / sources["volume_ml"]
print(sources.round(2).to_string(index=False))

`effective_volume_ml` is $(\sum w)^2/\sum w^2$ in mL: the volume a uniform source of the same concentration would fill. A low ratio to the mask volume means the activity is concentrated, as in the rest of body, where a small part of the volume holds most of the activity.

## 4. Dose matrix

$D_c$ averaged over each target mask, on the pooled PET-weighted batches. Rows are targets, columns are sources. Targets exclude lesion voxels, as the labels of notebook 11 do; `rest_of_body` as a target is label 14 only.

**The mean is mass-weighted,** $\\sum \\rho D / \\sum \\rho$, that is energy over mass. The plain voxel mean was tried first and is wrong for targets with gas: in a gas voxel the dose is energy over an almost zero mass, so a few voxels dominate (GI self-dose 1.2 to 4.4 times the reference, section 7). `targets.csv` holds the plain mean per batch, used only for the statistics of section 9; means are recomputed here from the pooled images.

In [ ]:
pet = targets_raw[targets_raw.variant == "pet"]
density_of = {s: M.density_map(M.load(s, "ct"), M.load(s, "organs")) for s in available}
masks_of = {s: M.target_masks(s) for s in available}
read_dose = lambda s, name: np.asanyarray(nib.load(DATA / s / f"{name}.nii.gz").dataobj).astype(np.float64)
rows = []
for s in available:
    for variant in ("pet", "uniform"):
        for c in COMPARTMENTS:
            dose = read_dose(s, f"D_{c}_{variant}")
            means = M.mass_weighted_on_targets(dose, density_of[s], masks_of[s])
            for t, value in means.items():
                rows.append(dict(study_id=s, variant=variant, compartment=c, target=t, D=value, zero_voxel_fraction=float((dose[masks_of[s][t]] == 0).mean()) if masks_of[s][t].any() else np.nan))
pooled_all = pd.DataFrame(rows)
pooled_all.to_csv(DATA / "pooled_targets_mass_weighted.csv", index=False)
pooled = pooled_all[pooled_all.variant == "pet"]
matrix = {s: pooled[pooled.study_id == s].pivot(index="target", columns="compartment", values="D").reindex(index=TARGETS, columns=COMPARTMENTS) for s in available}
for s, m in matrix.items():
    print(STUDY_NAME[s], "Gy/(MBq h)")
    print(m.to_string(float_format=lambda v: f"{v:.2e}"))
    m.to_csv(DATA / s / "dose_matrix_pet_gy_per_mbq_h.csv")

short = {"kidneys_functional": "kidneys", "salivary_glands": "glands", "gi_tract": "GI", "rest_of_body": "rest", "spine_L2_L4": "L2-L4", "blood_pool": "blood"}
label = lambda name: short.get(name, name)
fig, axes = plt.subplots(1, len(matrix), figsize=(4.3 * len(matrix), 4), squeeze=False)
for ax, (s, m) in zip(axes[0], matrix.items()):
    image = ax.imshow(np.log10(m.to_numpy(dtype=float)), cmap="Blues", vmin=-9, vmax=-1)
    ax.set_xticks(range(len(COMPARTMENTS)), [label(c) for c in COMPARTMENTS], rotation=45, ha="right")
    ax.set_yticks(range(len(TARGETS)), [label(t) for t in TARGETS])
    ax.set_xlabel("source"); ax.set_title(STUDY_NAME[s], loc="left")
    ax.spines[["left", "bottom"]].set_visible(False)
axes[0, 0].set_ylabel("target")
fig.colorbar(image, ax=axes, shrink=.75, label="log10 Gy/(MBq h)")
fig.savefig(QC / "dose_matrix.png", dpi=150, bbox_inches="tight"); plt.show()

**Reading.** The diagonal dominates: self-dose is 2-4 orders of magnitude above any cross-dose. With the test TIA, cross-doses are 1-4% of the dose of kidneys, liver, spleen, glands, bladder and small lesions on the typical phantom, and 8% for the GI tract (`cross_fraction`, section 9). They matter in three cases: bladder (26%) and glands and GI (11%) on the high-TLV phantom, where the lesions are the neighbours; red marrow and blood, whose dose comes from `rest_of_body`, because that source contains them.

Anatomy is visible in the cross-doses. In the single-kidney phantom the only kidney is the right one, so kidney←spleen falls to 1.0e-7 against 5.2e-7 (typical) and 7.6e-7 (high TLV), and kidney←liver rises to 7.8e-7 against 3.8e-7 and 2.9e-7.

## 5. Linearity

On the typical phantom, a direct run of 2×10⁶ decays with the combined source $\sum_c \mathrm{TIA}_c w_c$, against $\sum_c \mathrm{TIA}_c D_c$. The TIA vector is only a test case: 7.4 GBq × PET %IA of the compartment × $T_\mathrm{eff}/\ln 2$ with assumed $T_\mathrm{eff}$ (`combined_tia.csv`), not a PBPK draw.

The expected noise of the difference comes from the batches: the batch spread of each $D_c$, scaled to the decays each compartment gets in the pooled and in the direct run.

In [ ]:
batch_sd = pet.groupby(["study_id", "compartment", "target"]).gy_per_mbq_h.std(ddof=1).rename("sd").reset_index()
n_batches = pet.groupby(["study_id", "compartment"]).batch.nunique()
study = R.COMBINED_STUDY
linearity = None
if (DATA / study / "D_combined_direct_gy.nii.gz").exists():
    tia = pd.read_csv(DATA / study / "combined_tia.csv").set_index("compartment")
    print(tia.round(3).to_string())
    total = tia.tia_mbq_h.sum()
    direct = pd.Series(M.mass_weighted_on_targets(read_dose(study, "D_combined_direct_gy"), density_of[study], masks_of[study]))
    rows = []
    for target in TARGETS:
        predicted = sum(tia.loc[c, "tia_mbq_h"] * matrix[study].loc[target, c] for c in COMPARTMENTS)
        variance = 0.0
        for c in COMPARTMENTS:
            sd = batch_sd[(batch_sd.study_id == study) & (batch_sd.compartment == c) & (batch_sd.target == target)].sd.iloc[0]
            k = n_batches.loc[(study, c)]
            n_direct = R.COMBINED_N * tia.loc[c, "tia_mbq_h"] / total
            variance += (tia.loc[c, "tia_mbq_h"] * sd) ** 2 * (1 / k + R.BATCH_SIZE / n_direct)
        rows.append(dict(target=target, predicted_gy=predicted, direct_gy=direct[target], ratio=predicted / direct[target], expected_noise_pct=100 * np.sqrt(variance) / direct[target]))
    linearity = pd.DataFrame(rows)
    linearity["z"] = (linearity.ratio - 1) * 100 / linearity.expected_noise_pct
    linearity.to_csv(DATA / "linearity_check.csv", index=False)
    print(linearity.round(4).to_string(index=False))

    d_sum = sum(tia.loc[c, "tia_mbq_h"] * np.asanyarray(nib.load(DATA / study / f"D_{c}_pet.nii.gz").dataobj).astype(np.float64) for c in COMPARTMENTS)
    d_direct = np.asanyarray(nib.load(DATA / study / "D_combined_direct_gy.nii.gz").dataobj).astype(np.float64)
    organs = M.load(study, "organs")
    body = organs > 0
    block = 4
    trim = tuple(n - n % block for n in d_sum.shape)
    coarse = lambda a: a[:trim[0], :trim[1], :trim[2]].reshape(trim[0] // block, block, trim[1] // block, block, trim[2] // block, block).sum(axis=(1, 3, 5))
    weight = coarse(body.astype(float))
    full = weight == block ** 3
    a, b = coarse(d_sum * body)[full] / block ** 3, coarse(d_direct * body)[full] / block ** 3
    hot = b > np.percentile(b, 99)
    print(f"Blocks of {block} voxels inside the body: {full.sum()}; r = {np.corrcoef(a, b)[0, 1]:.4f}; hottest 1%: r = {np.corrcoef(a[hot], b[hot])[0, 1]:.3f}, median sum/direct {np.median(a[hot] / b[hot]):.3f}; all blocks median {np.median(a / b):.3f}")

**Reading. Linearity holds:** $\sum \mathrm{TIA}_c D_c$ agrees with the direct run within 1.1% on every target, and on blocks of 4 voxels $r$ = 0.995 (median ratio 0.999; 0.994 in the hottest 1%). Liver and spleen are 2.4 and 3.1 expected standard deviations off, with ratios 1.00 and 1.01. Batch spreads from 5 batches are uncertain by about a third, so this does not establish a bias, but one below 1% is not excluded. For this purpose it does not matter.

## 6. VSV against Monte Carlo

The VSV of notebook 04: GATE kernel in water, 4 mm, truncated at about ±40 mm. Each source is moved to the 4 mm grid by trilinear deposit (total conserved exactly), convolved, interpolated back to the PET grid and divided by the local density of the same material map (notebook 05: VSV/ρ). Water values are kept for comparison. The kernel has no contribution beyond its edge, so cross-doses between distant organs are expected to be missing.

In [ ]:
kernel = np.load(M.KERNEL)
cache = DATA / "vsv_targets.csv"
old = pd.read_csv(cache) if cache.exists() else pd.DataFrame(columns=["study_id", "compartment", "variant"])
seen = set(zip(old.study_id, old.compartment, old.variant))
rows = []
for study in available:
    organs, ct = M.load(study, "organs"), M.load(study, "ct")
    density = density_of[study]
    masks = masks_of[study]
    spacing = M.spacing_of(study)
    for variant in ("pet", "uniform"):
        for c in COMPARTMENTS:
            if (study, c, variant) in seen or not (DATA / study / f"D_{c}_{variant}.nii.gz").exists():
                continue
            start = time.time()
            rho, water = M.vsv_dose(M.source_weights(study, c, variant), spacing, density, kernel)
            seconds = time.time() - start
            mean_rho, mean_water = M.mass_weighted_on_targets(rho, density, masks), M.mass_weighted_on_targets(water, density, masks)
            for t in masks:
                rows.append(dict(study_id=study, compartment=c, variant=variant, target=t, vsv_rho=mean_rho[t], vsv_water=mean_water[t], vsv_seconds=seconds))
if rows:
    old = pd.concat([old, pd.DataFrame(rows)], ignore_index=True)
    old.to_csv(cache, index=False)
vsv = old
mc_by_variant = pooled_all.rename(columns={"D": "mc"})[["study_id", "compartment", "variant", "target", "mc"]]
compare = mc_by_variant.merge(vsv, on=["study_id", "compartment", "variant", "target"])
compare["ratio"] = compare.vsv_rho / compare.mc
compare["ratio_water"] = compare.vsv_water / compare.mc
self_rows = compare[compare.compartment == compare.target]
print("Self-dose, VSV/rho over MC")
print(self_rows.assign(phantom=self_rows.study_id.map(STUDY_NAME)).pivot_table(index="compartment", columns=["variant", "phantom"], values="ratio").reindex(COMPARTMENTS).round(3).to_string())
print("\nVSV in water over MC, PET source")
print(self_rows[self_rows.variant == "pet"].assign(phantom=lambda d: d.study_id.map(STUDY_NAME)).pivot_table(index="compartment", columns="phantom", values="ratio_water").reindex(COMPARTMENTS).round(3).to_string())
for s in available:
    m = compare[(compare.study_id == s) & (compare.variant == "pet")].pivot(index="target", columns="compartment", values="ratio").reindex(index=TARGETS, columns=COMPARTMENTS)
    print("\nVSV/rho over MC, all pairs, PET source,", STUDY_NAME[s])
    print(m.round(2).to_string())
compare.to_csv(DATA / "vsv_vs_mc.csv", index=False)

rows = []
for s in available:
    for c in COMPARTMENTS:
        dose = read_dose(s, f"D_{c}_pet")
        blurred = M.through_kernel_grid(dose * density_of[s], M.spacing_of(s)).astype(np.float64)
        mask = masks_of[s][c]
        rows.append(dict(phantom=STUDY_NAME[s], compartment=c, resampling_blur=float(blurred[mask].sum() / (dose[mask] * density_of[s][mask]).sum())))
blur = pd.DataFrame(rows).merge(self_rows[self_rows.variant == "pet"].assign(phantom=lambda d: d.study_id.map(STUDY_NAME))[["phantom", "compartment", "ratio"]], on=["phantom", "compartment"])
blur = blur.rename(columns={"ratio": "vsv_rho_over_mc"})
blur["left_after_blur"] = blur.vsv_rho_over_mc / blur.resampling_blur
print("\nMC self-dose after the round trip to the 4 mm grid, no kernel, over MC")
print(blur.pivot(index="compartment", columns="phantom", values="resampling_blur").reindex(COMPARTMENTS).round(3).to_string())
print("\nVSV/rho over MC, divided by that blur")
print(blur.pivot(index="compartment", columns="phantom", values="left_after_blur").reindex(COMPARTMENTS).round(3).to_string())

def realistic_tia(study):
    pet_image, organs = M.load(study, "pet_pct_ia_ml").astype(np.float64), M.load(study, "organs")
    voxel_ml = np.prod(M.spacing_of(study)) / 1000
    return {c: R.TEST_ACTIVITY_MBQ * pet_image[np.isin(organs, labels)].sum() * voxel_ml / 100 * R.TEST_T_EFF_H[c] / np.log(2) for c, labels in M.COMPARTMENTS.items()}

rows = []
for s in available:
    tia = realistic_tia(s)
    g = compare[(compare.study_id == s) & (compare.variant == "pet")].set_index(["compartment", "target"])
    for t in TARGETS:
        mc_total = sum(tia[c] * g.loc[(c, t), "mc"] for c in COMPARTMENTS)
        vsv_total = sum(tia[c] * g.loc[(c, t), "vsv_rho"] for c in COMPARTMENTS)
        rows.append(dict(phantom=STUDY_NAME[s], target=t, mc_gy=mc_total, vsv_rho_gy=vsv_total, ratio=vsv_total / mc_total, rest_share_mc=tia["rest_of_body"] * g.loc[("rest_of_body", t), "mc"] / mc_total, rest_share_vsv=tia["rest_of_body"] * g.loc[("rest_of_body", t), "vsv_rho"] / vsv_total))
total_vsv = pd.DataFrame(rows)
total_vsv.to_csv(DATA / "vsv_vs_mc_total_dose.csv", index=False)
print("\nTotal target dose with the test TIA, VSV/rho over MC")
print(total_vsv.pivot(index="target", columns="phantom", values="ratio").reindex(TARGETS).round(3).to_string())
print("\nShare of the rest-of-body source in the target dose: MC, VSV/rho")
print(total_vsv.set_index(["phantom", "target"])[["rest_share_mc", "rest_share_vsv"]].unstack(0).reindex(TARGETS).round(3).to_string())

In [ ]:
markers = {"pet": "o", "uniform": "s"}
fig, ax = plt.subplots(figsize=(7, 3.4))
order = COMPARTMENTS
for k, s in enumerate(available):
    g = self_rows[(self_rows.study_id == s) & (self_rows.variant == "pet")].set_index("compartment").reindex(order)
    ax.scatter(np.arange(len(order)) + (k - 1) * .15, g.ratio, s=42, color=STUDY_COLOR[s], edgecolor="white", linewidth=1.2, label=STUDY_NAME[s], zorder=3)
ax.axhline(1, color=INK, lw=1)
ax.set_xticks(range(len(order)), [label(c) for c in order])
ax.set(ylabel="VSV/ρ self-dose ÷ Monte Carlo", title="Self-dose per unit TIA, PET-weighted source")
ax.grid(axis="y", color=GRID, lw=.6); ax.set_axisbelow(True); ax.legend(frameon=False, ncol=1, loc="lower left")
fig.tight_layout(); fig.savefig(QC / "vsv_self_ratio.png", dpi=150); plt.show()

study = available[0]
if (DATA / study / "D_kidneys_functional_pet.nii.gz").exists():
    organs, ct = M.load(study, "organs"), M.load(study, "ct")
    spacing = M.spacing_of(study)
    rho, _ = M.vsv_dose(M.source_weights(study, "kidneys_functional", "pet"), spacing, M.density_map(ct, organs), kernel)
    mc = np.asanyarray(nib.load(DATA / study / "D_kidneys_functional_pet.nii.gz").dataobj)
    kidney = organs == 1
    x, y, _ = [int(np.median(i)) for i in np.where(kidney)]
    z = (np.arange(mc.shape[2]) - np.median(np.where(kidney)[2])) * spacing[2]
    window = lambda a: a[x - 4:x + 5, y - 4:y + 5, :].mean(axis=(0, 1))
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.semilogy(z, window(mc), color=STUDY_COLOR[study], lw=2, label="Monte Carlo")
    ax.semilogy(z, np.maximum(window(rho), 1e-12), color="#eb6834", lw=2, label="VSV/ρ")
    ax.set(xlabel="axial distance from the kidney centre (mm)", ylabel="Gy/(MBq h)", ylim=(1e-9, None), title="Kidney source, axial profile through the kidney: VSV stops at the kernel edge")
    ax.grid(color=GRID, lw=.6); ax.legend(frameon=False)
    fig.tight_layout(); fig.savefig(QC / "kidney_profile.png", dpi=150); plt.show()

**Reading.**
- **The self-dose deficit of VSV/ρ is the 4 mm grid, not the kernel physics.** VSV/ρ is 0.91-0.98 for kidneys, liver and spleen, 0.76-0.85 for glands, 0.81-0.85 for lesions. Sending the Monte Carlo dose itself to the 4 mm grid and back, with no kernel, gives almost the same loss: the ratio left is 0.97-1.01 for everything except the rest of body (0.96). So this is partial volume of the resampling. Notebook 05 did not see it (VSV/ρ 0.997-1.017) because there Monte Carlo was also scored on the 4 mm grid.
- **The density correction goes the expected way only for organs.** Water VSV is 0.96-1.03 on organs, and dividing by ρ lowers it by the density.
- **Cross-doses are not usable.** Near organs are mostly at 0.1-0.6 of Monte Carlo and far ones at 0, as the profile shows: the kernel ends at ±40 mm while the photon tail continues for hundreds of millimetres.
- **The rest-of-body source goes the other way:** 1.5-6.9 times Monte Carlo on organs, because the blur carries its PET halo (kidney and gland spill-out) into the masks. In VSV this source gives 17-32% of the gland dose, against 2-8% in Monte Carlo.
- **On the total dose the errors partly cancel:** with the test TIA, kidneys, liver, spleen and glands are at 0.93-1.03 of Monte Carlo, **but** GI 0.87-0.94, lesions 0.84-0.90, red marrow 0.69-1.06, blood 0.87-0.93 and the bladder of the high-TLV phantom 1.35. The agreement comes from compensation, so it cannot be trusted on a new TIA vector or a new anatomy.

## 7. Organ S-values

**Self-dose.** From EANM 2022 (Sjögreen Gleisner et al.), Tables 2 and 5, IDAC-Dose 2.1 adult male, read in the full text: S(kidney←kidney) 0.204 mGy/(MBq·h) at 422 g, S(liver←liver) 0.0376 at 2360 g, S(spleen←spleen) 0.377 at 228.4 g, S(salivary←salivary) 0.947 at 88.98 g. Scaled to the patient by mass, $S_\mathrm{ref}\, m_\mathrm{ref}/m$ (Eq. 4 of that paper). Local energy deposition $\Delta_e/m$ with $\Delta_e$ = 0.0853 mJ/(MBq·h), ICRP 107, for every compartment, including those without a table value. Uniform source: this is what an S-value assumes.

**Cross-dose.** Two checks.
- **OpenDose** (Chauvin et al., J Nucl Med 2020), ICRP 110 adult male, Lu-177, ICRP 107 decay data, downloaded by `scripts/dataset/fetch_opendose_svalues.py` for 10 sources (`opendose_lu177_icrp110am.csv`, licence CC BY-SA 4.0, research use). Mapping below; values are compared unscaled, because a cross-dose depends on the distance between organs, which differs from patient to patient. OpenDose is one reference anatomy, so a factor of order 1.5-2 for far pairs is anatomy and not disagreement. Self-doses are compared after mass scaling.
- **Reciprocity:** for uniform sources in a uniform medium the specific absorbed fraction is symmetric, so $D(t \leftarrow s) = D(s \leftarrow t)$ per unit TIA for photons; a ratio far from 1 flags a geometry or statistical problem.

| Ours | OpenDose source | OpenDose target |
|---|---|---|
| `kidneys_functional`, `liver`, `spleen`, `salivary_glands` | RKidney + LKidney, Liver, Spleen, Salivary glands | same |
| `bladder` | Urinary bladder contents | contents + wall, mass-weighted |
| `gi_tract` | stomach wall, small intestine wall, colon, mass-weighted | same |
| `spine_L2_L4` | - | Lumbar spine, spongiosa + cortical, mass-weighted |
| `blood_pool` | - | Blood vessels, trunk |

In [ ]:
EANM = {"kidneys_functional": (0.204, 422.0), "liver": (0.0376, 2360.0), "spleen": (0.377, 228.4), "salivary_glands": (0.947, 88.98)}
DELTA_E_MJ = 0.0853
uniform = pooled_all[pooled_all.variant == "uniform"].pivot_table(index=["study_id", "target"], columns="compartment", values="D")
pet_mean = pooled_all[pooled_all.variant == "pet"].pivot_table(index=["study_id", "target"], columns="compartment", values="D")
rows, masses = [], {}
for study in available:
    organs, ct = M.load(study, "organs"), M.load(study, "ct")
    density = M.density_map(ct, organs)
    voxel_ml = np.prod(M.spacing_of(study)) / 1000
    masses[study] = {name: float(density[np.isin(organs, labels)].sum() * voxel_ml) for name, labels in M.COMPARTMENTS.items()}
    for c in COMPARTMENTS:
        if (study, c) not in uniform.index or c not in uniform.columns or c == "rest_of_body":
            continue
        m = masses[study][c]
        row = dict(phantom=STUDY_NAME[study], compartment=c, mass_g=m, led=DELTA_E_MJ / m)
        row["mc_uniform"] = uniform.loc[(study, c), c]
        row["mc_pet"] = pet_mean.loc[(study, c), c]
        row["s_mass_scaled"] = EANM[c][0] * EANM[c][1] / m * 1e-3 if c in EANM else np.nan
        rows.append(row)
self_s = pd.DataFrame(rows)
self_s["mc_over_s"] = self_s.mc_uniform / self_s.s_mass_scaled
self_s["mc_over_led"] = self_s.mc_uniform / self_s.led
self_s["mc_pet_over_uniform"] = self_s.mc_pet / self_s.mc_uniform
self_s.to_csv(DATA / "self_dose_vs_s_values.csv", index=False)
print(self_s.to_string(index=False, float_format=lambda v: f"{v:.3g}"))

pairs = [("kidneys_functional", "liver"), ("kidneys_functional", "spleen"), ("liver", "spleen"), ("liver", "gi_tract"), ("kidneys_functional", "gi_tract"), ("spleen", "gi_tract"), ("bladder", "gi_tract"), ("liver", "rest_of_body"), ("kidneys_functional", "bladder"), ("liver", "salivary_glands")]
rows = []
for study in available:
    for a, b in pairs:
        if (study, a) not in uniform.index or (study, b) not in uniform.index or a not in uniform.columns or b not in uniform.columns:
            continue
        ta, tb = M.COMPARTMENTS[a], M.COMPARTMENTS[b]
        rows.append(dict(phantom=STUDY_NAME[study], s=a, t=b, D_t_from_s=uniform.loc[(study, b), a], D_s_from_t=uniform.loc[(study, a), b], reciprocity=uniform.loc[(study, b), a] / uniform.loc[(study, a), b]))
reciprocity = pd.DataFrame(rows)
reciprocity.to_csv(DATA / "reciprocity.csv", index=False)
print(reciprocity.to_string(index=False, float_format=lambda v: f"{v:.3g}"))

opendose = pd.read_csv(DATA / "opendose_lu177_icrp110am.csv")
stats_rel = pet.groupby(["study_id", "compartment", "target"]).gy_per_mbq_h.agg(lambda v: v.std(ddof=1) / v.mean()).rename("batch_rel_sd").reset_index()
GI_PARTS = ["Stomach wall", "Small intestine wall", "Colon"]
OD_SOURCE = {"kidneys_functional": ["RKidney + LKidney"], "liver": ["Liver"], "spleen": ["Spleen"], "salivary_glands": ["Salivary glands"], "bladder": ["Urinary bladder contents"], "gi_tract": GI_PARTS}
OD_TARGET = {"kidneys_functional": ["RKidney + LKidney"], "liver": ["Liver"], "spleen": ["Spleen"], "salivary_glands": ["Salivary glands"], "bladder": ["Urinary bladder contents", "Urinary bladder wall"], "gi_tract": GI_PARTS, "spine_L2_L4": ["Lumbar spine, spongiosa", "Lumbar spine, cortical"], "blood_pool": ["Blood vessels, trunk"]}
mass_od = opendose.drop_duplicates("target").set_index("target").target_mass_g
s_od = opendose.set_index(["source", "target"])

def od_value(source, target):
    srcs, tgts = OD_SOURCE[source], OD_TARGET[target]
    total, error = 0.0, 0.0
    for a in srcs:
        for b in tgts:
            w = mass_od[a] / sum(mass_od[x] for x in srcs) * mass_od[b] / sum(mass_od[x] for x in tgts)
            total += w * s_od.loc[(a, b), "s_mgy_per_mbq_s"]
            error += (w * s_od.loc[(a, b), "standard_error"]) ** 2
    return total * M.MGY_MBQ_S_TO_GY_MBQ_H, np.sqrt(error) * M.MGY_MBQ_S_TO_GY_MBQ_H

rows = []
for study in available:
    for source in OD_SOURCE:
        for target in OD_TARGET:
            if (study, target) not in uniform.index or source not in uniform.columns:
                continue
            value, error = od_value(source, target)
            mc = uniform.loc[(study, target), source]
            is_self = source == target
            scale = sum(mass_od[x] for x in OD_TARGET[target]) / masses[study][target] if is_self else 1.0
            rows.append(dict(phantom=STUDY_NAME[study], source=source, target=target, kind="self" if is_self else "cross", mc_uniform=mc, opendose=value, opendose_rel_se=error / value, mass_scale=scale, mc_over_opendose=mc / (value * scale)))
od_compare = pd.DataFrame(rows)
od_compare = od_compare.merge(stats_rel.assign(phantom=stats_rel.study_id.map(STUDY_NAME)).rename(columns={"compartment": "source"})[["phantom", "source", "target", "batch_rel_sd"]], on=["phantom", "source", "target"], how="left")
od_compare["reliable"] = (od_compare.opendose_rel_se < 0.05) & (od_compare.batch_rel_sd < 0.10)
od_compare.to_csv(DATA / "mc_vs_opendose.csv", index=False)
print("Self-dose, Monte Carlo (uniform) over mass-scaled OpenDose")
print(od_compare[od_compare.kind == "self"].pivot(index="source", columns="phantom", values="mc_over_opendose").round(3).to_string())
for phantom, g in od_compare[od_compare.kind == "cross"].groupby("phantom"):
    print("\nCross-dose, Monte Carlo (uniform) over OpenDose,", phantom, "(rows target, columns source)")
    print(g.pivot(index="target", columns="source", values="mc_over_opendose").reindex(index=list(OD_TARGET), columns=list(OD_SOURCE)).round(2).to_string())
print("\nLargest OpenDose relative SE among the values used:", od_compare.opendose_rel_se.round(3).max(), "| pairs above 5%:", od_compare.loc[od_compare.opendose_rel_se > .05, ["source", "target"]].drop_duplicates().values.tolist())
good = od_compare[(od_compare.kind == "cross") & od_compare.reliable]
print(f"Cross-dose pairs with OpenDose SE < 5% and MC batch SD < 10%: {len(good)} of {(od_compare.kind == 'cross').sum()}; MC/OpenDose quartiles {good.mc_over_opendose.quantile([.25, .5, .75]).round(2).tolist()}; inside a factor of 2: {good.mc_over_opendose.between(.5, 2).mean():.2f}")
print(good.groupby("phantom").mc_over_opendose.median().round(2).to_dict())

fig, ax = plt.subplots(figsize=(4.6, 4.3))
for study in available:
    g = od_compare[(od_compare.phantom == STUDY_NAME[study]) & (od_compare.kind == "cross") & od_compare.reliable]
    ax.loglog(g.opendose, g.mc_uniform, "o", ms=6, color=STUDY_COLOR[study], mec="white", mew=.8, label=STUDY_NAME[study])
lim = [1e-9, 2e-5]
ax.plot(lim, lim, color=INK, lw=1); ax.plot(lim, [2 * x for x in lim], color=INK, lw=.6, ls=":"); ax.plot(lim, [x / 2 for x in lim], color=INK, lw=.6, ls=":")
ax.set(xlim=lim, ylim=lim, xlabel="OpenDose ICRP 110 AM, Gy/(MBq h)", ylabel="Monte Carlo, uniform source, Gy/(MBq h)", title="Cross-doses (dotted: factor of 2)")
ax.grid(color=GRID, lw=.6); ax.legend(frameon=False)
fig.tight_layout(); fig.savefig(QC / "cross_dose_vs_opendose.png", dpi=150); plt.show()

**Reading.**
- **Self-dose: Monte Carlo agrees with both references within about 1%.** Over mass-scaled IDAC-Dose (EANM) 0.995-1.01 on kidneys, liver, spleen and glands; over mass-scaled OpenDose 0.987-1.01 on these and on the bladder. The GI tract is at 1.04-1.10 with a coarse mapping (walls only, while our mask holds wall, contents and gas). The two tables use different organ masses (OpenDose kidneys 310 g, liver 1800 g, spleen 150 g; IDAC 422, 2360, 228 g), and mass scaling absorbs that, as expected. Over local deposition: organs 1.01-1.05, the excess being photons; glands 0.99-1.00, as electrons escape from small structures.
- **Cross-dose: agreement of order 1 with a real anatomical spread.** On the 77 of 126 pairs with OpenDose SE below 5% and batch spread below 10%, Monte Carlo/OpenDose has a median of 0.85 (quartiles 0.67-1.11) and 86% lie within a factor of 2. Medians per phantom are 0.73 (typical), 0.88 (single kidney), 0.96 (high TLV); the reason for the low typical value was not studied. Large deviations have an anatomical reason where one can be named: 0.14-0.16 for kidney←spleen on the single-kidney phantom, whose left kidney is missing. Blood pool from the bladder (0.08-0.18) is not a fair pair: our target is the aorta, OpenDose's includes the pelvic vessels. Pairs with the glands and the bladder are too noisy to say anything.
- **OpenDose errors:** a few published standard errors are larger than the value (liver→bladder, up to 28 times); those pairs are excluded.
- **Reciprocity:** 25 of 30 ratios within 0.8-1.25. The others involve small photon signals from one uniform batch. Liver←→rest of body is steadily 0.89-0.90 because source and target are not the same set of labels.

## 8. PET-weighted against uniform source

Same compartment, same target. The ratio shows how much the real distribution inside the compartment matters: a network trained with uniform per-organ sources would miss it. The uniform run has one batch, so its noise dominates for far targets.

In [ ]:
pu = pooled_all.pivot_table(index=["study_id", "compartment", "target"], columns="variant", values="D").reset_index()
pu["ratio"] = pu.pet / pu.uniform
for s in available:
    print(STUDY_NAME[s], "PET-weighted / uniform")
    print(pu[pu.study_id == s].pivot(index="target", columns="compartment", values="ratio").reindex(index=TARGETS, columns=COMPARTMENTS).round(2).to_string())
pu.to_csv(DATA / "pet_vs_uniform.csv", index=False)

**Reading.** For **self-dose the distribution does not matter at organ level:** PET/uniform = 0.95-1.02 for every compartment in every phantom, because electrons deposit locally and the mean depends only on the mass. It will matter at voxel level, which is why the PET-weighted source is the one to keep. For **cross-doses** ratios are near 1 on near, well-sampled pairs; those with the glands, bladder and small lesions are mostly noise from the single uniform batch. **The real effect is the rest-of-body source**: 1.1-6.9 times uniform on organs (kidneys 6.3, 1.9, 3.9), since its activity sits in the halo around the organs. It is mostly spill-out of the organ activity, not rest-of-body activity. It is about 1% of the kidney dose in Monte Carlo, so the effect is small, but it does not disappear.

## 9. Statistical uncertainty and runtime against the number of primaries

Batches give the sample standard deviation of each target mean at $n_b$ = 2×10⁵ decays; the standard error at $n$ decays is $\sigma \sqrt{n_b / n}$. This assumes $1/\sqrt{n}$, tested below on the typical phantom by the spread of pooled pairs. With 3-5 batches the estimate itself is uncertain by about ±30-50%.

What matters for a phantom is not each $D_c$ but the dose of a target in a realistic TIA vector, so the last table adds the terms in quadrature, with the test TIA built like in section 5 for each phantom and $n$ decays for **each** compartment.

In [ ]:
stats = pet.groupby(["study_id", "compartment", "target"]).gy_per_mbq_h.agg(mean="mean", sd=lambda v: v.std(ddof=1), batches="count").reset_index()
stats["rel_sd_per_2e5"] = stats.sd / stats["mean"]
stats["n_for_1pct"] = R.BATCH_SIZE * (stats.rel_sd_per_2e5 / 0.01) ** 2
stats.to_csv(DATA / "statistics_per_target.csv", index=False)
for s in available:
    print(STUDY_NAME[s], "relative SE (%) of the target mean at 1e6 decays")
    t = stats[stats.study_id == s].assign(se=lambda d: 100 * d.rel_sd_per_2e5 * np.sqrt(R.BATCH_SIZE / 1e6))
    print(t.pivot(index="target", columns="compartment", values="se").reindex(index=TARGETS, columns=COMPARTMENTS).round(2).to_string())

check = pet[(pet.study_id == R.COMBINED_STUDY)]
pair = check[check.batch.isin([0, 1])].groupby(["compartment", "target"]).gy_per_mbq_h.mean().rename("a").to_frame()
pair["b"] = check[check.batch.isin([2, 3])].groupby(["compartment", "target"]).gy_per_mbq_h.mean()
pair = pair.join(stats[stats.study_id == R.COMBINED_STUDY].set_index(["compartment", "target"])[["rel_sd_per_2e5", "mean"]])
if len(check.batch.unique()) >= 4:
    pair = pair[pair.a.notna() & pair.b.notna()]
    pair["rel_diff"] = (pair.a - pair.b) / pair["mean"] / np.sqrt(2) / (pair.rel_sd_per_2e5 / np.sqrt(2))
    print("\n(pooled of 2 - pooled of 2) / expected, median |z|:", round(pair.rel_diff.abs().median(), 2), "(0.67 expected for normal noise)")

rows = []
for s in available:
    tia = realistic_tia(s)
    g = stats[stats.study_id == s].set_index(["compartment", "target"])
    for t in TARGETS:
        contributions = {c: tia[c] * g.loc[(c, t), "mean"] for c in COMPARTMENTS if (c, t) in g.index}
        variance = sum((tia[c] * g.loc[(c, t), "sd"]) ** 2 for c in contributions)
        dose = sum(contributions.values())
        main = max(contributions, key=contributions.get)
        cross = 1 - contributions.get(t, 0.0) / dose if t in contributions else np.nan
        row = dict(phantom=STUDY_NAME[s], target=t, test_dose_gy=dose, main_source=label(main), cross_fraction=cross)
        for n in (5e4, 2e5, 5e5, 1e6, 5e6):
            row[f"se_pct_{n:.0e}"] = 100 * np.sqrt(variance * R.BATCH_SIZE / n) / dose
        rows.append(row)
total_se = pd.DataFrame(rows)
total_se.to_csv(DATA / "dose_uncertainty_vs_n.csv", index=False)
print(total_se.to_string(index=False, float_format=lambda v: f"{v:.3g}"))

voxel = targets_raw[(targets_raw.variant == "pet") & (targets_raw.compartment == targets_raw.target)]
print("\nMedian per-voxel relative uncertainty of a single 2e5 batch, self-dose (GATE; a voxel with no hit counts as 0):")
print(voxel.assign(phantom=voxel.study_id.map(STUDY_NAME)).pivot_table(index="compartment", columns="phantom", values="voxel_relative_uncertainty_median").reindex(COMPARTMENTS).round(3).to_string())
own = pooled_all[(pooled_all.variant == "pet") & (pooled_all.compartment == pooled_all.target)]
print("\nFraction of target voxels with no dose at all in the pooled self-dose image:")
print(own.assign(phantom=own.study_id.map(STUDY_NAME)).pivot_table(index="compartment", columns="phantom", values="zero_voxel_fraction").reindex(COMPARTMENTS).round(3).to_string())
organs_body = {s: int((M.load(s, "organs") > 0).sum()) for s in available}
print("Decays per body voxel, rest-of-body source, pooled:", {STUDY_NAME[s]: round(n_batches.loc[(s, "rest_of_body")] * R.BATCH_SIZE / organs_body[s], 2) for s in available})
s0 = R.COMBINED_STUDY
mc_map = read_dose(s0, "D_combined_direct_gy")
tia0 = realistic_tia(s0)
vsv_map = sum(tia0[c] * M.vsv_dose(M.source_weights(s0, c, "pet"), M.spacing_of(s0), density_of[s0], kernel)[0].astype(np.float64) for c in COMPARTMENTS)
body0 = M.load(s0, "organs") > 0
keep = body0 & (density_of[s0] > 0.5)
for name, mask in (("kidneys", masks_of[s0]["kidneys_functional"]), ("liver", masks_of[s0]["liver"]), ("spleen", masks_of[s0]["spleen"])):
    m = mask & keep
    print(f"{name}: voxel r(VSV/rho, direct MC 2e6) = {np.corrcoef(vsv_map[m], mc_map[m])[0, 1]:.3f}, CV of MC voxels {mc_map[m].std() / mc_map[m].mean():.2f}, CV of VSV voxels {vsv_map[m].std() / vsv_map[m].mean():.2f}")

In [ ]:
s = available[0]
fig, ax = plt.subplots(figsize=(7, 3.6))
ns = np.geomspace(1e4, 1e7, 30)
tg = total_se[total_se.phantom == STUDY_NAME[s]].set_index("target")
palette = {"kidneys_functional": "#2a78d6", "salivary_glands": "#eb6834", "spine_L2_L4": "#1baf7a", "lesions": "#6250d6"}
for t, color in palette.items():
    ax.loglog(ns, tg.loc[t, "se_pct_1e+06"] * np.sqrt(1e6 / ns), lw=2, color=color)
    ax.annotate(label(t), (ns[-1], tg.loc[t, "se_pct_1e+06"] * np.sqrt(1e6 / ns[-1])), xytext=(4, 0), textcoords="offset points", va="center", color=INK)
ax.axhline(1, color=INK, lw=.8, ls=":")
ax.set(xlabel="decays per compartment", ylabel="standard error of the target dose (%)", title=f"Target dose with the test TIA: statistical error, {STUDY_NAME[s]} phantom")
ax.grid(color=GRID, lw=.6, which="both"); ax.set_axisbelow(True)
fig.tight_layout(); fig.savefig(QC / "uncertainty_vs_n.png", dpi=150); plt.show()

run = targets_raw.drop_duplicates(["study_id", "compartment", "variant", "batch"]).copy()
run["voxels"] = run.study_id.map(lambda s: int(np.prod(grid_of(s)["shape"])))
run["rate_decays_per_s"] = run.n / run.seconds
print(run[run.compartment != "combined"].groupby(["study_id", "compartment"]).rate_decays_per_s.median().unstack(0).rename(columns=STUDY_NAME).round(0).to_string())
print(run[(run.variant == "timing") | (run.compartment == "combined")][["study_id", "compartment", "n", "seconds", "wall_seconds", "voxels"]].round(1).to_string(index=False))
run.to_csv(DATA / "runtime_per_run.csv", index=False)

**Reading.**
- **Organ doses converge quickly.** Self-dose SE is at most 0.35% at 10⁶ decays. With the test TIA and 2×10⁵ decays per compartment, the target dose has SE below 0.8% on kidneys, liver, spleen and most other organs; the worst are blood and L2-L4 (1.4-2.8%) and the glands and bladder of the high-TLV phantom (1.9%, 1.3%), which receive little. At 5×10⁵ everything is below 1.8%. Far cross-doses are noisy (20-85% at 10⁶ on the glands) but do not count, because their share of the dose is negligible.
- **The $1/\sqrt{n}$ scaling is compatible with the data:** split-pair median $|z|$ 0.81 against 0.67 expected, from few batches.
- **Voxel maps do not converge at this budget.** In a single 2×10⁵ batch the median per-voxel uncertainty is 9-29% in kidneys and spleen and 30-69% in liver and GI, on self-dose alone. At 5×10⁵ divide by 1.6. For the rest of body, the pooled image leaves 24-53% of voxels with no dose at all: 0.4-1 decays per body voxel. Direct run at 2×10⁶: voxel correlation with VSV/ρ is only 0.49-0.74, and the Monte Carlo CV inside an organ (0.37-0.53) exceeds the VSV one (0.27-0.30), mainly noise.
- **Speed depends on the grid:** about 3200 decays/s at 17 million voxels, about 6000 at 4.7 million, 4 threads. Initialisation is short, 18-25 s.

## 10. Cost for the training split and proposal

Wall time of one run: $t = t_\mathrm{init}(\text{grid}) + n / r(\text{grid})$, at 4 threads, with $t_\mathrm{init}$ from the runs of 10⁴ decays and $r$ from the 2×10⁵ batches. Both are interpolated linearly in the number of voxels between the two grid sizes of the pilot, so grids of the split that fall between them are estimates, and those above 17 million voxels a slight extrapolation. One phantom needs 8 PET-weighted runs. Disk comes from the files on disk. VSV time is measured in section 6, on one core.

In [ ]:
timing = run[run.variant == "timing"].set_index("study_id")
if len(timing) == len(R.STUDIES):
    per_batch = run[(run.variant == "pet") & (run.compartment != "combined")]
    rate = {s: (R.BATCH_SIZE - R.TIMING_N) / (per_batch[per_batch.study_id == s].seconds.mean() - timing.loc[s, "seconds"]) for s in R.STUDIES}
    init = {s: timing.loc[s, "wall_seconds"] - R.TIMING_N / rate[s] for s in R.STUDIES}
    print({STUDY_NAME[s]: (round(rate[s]), round(init[s], 1)) for s in R.STUDIES}, "(decays/s at 4 threads, init s)")
    voxels = train.study_id.map(lambda s: int(np.prod(grid_of(s)["shape"]))).to_numpy()
    grids = pd.Series(voxels).value_counts()
    ref = [np.prod(grid_of(s)["shape"]) for s in R.STUDIES]
    per_decay = np.polyfit(ref, [1 / rate[s] for s in R.STUDIES], 1)
    fit = np.polyfit(ref, [init[s] for s in R.STUDIES], 1)
    rows = []
    for n in (5e4, 1e5, 2e5, 5e5, 1e6, 5e6):
        one = 8 * (np.polyval(fit, voxels) + n * np.polyval(per_decay, voxels))
        rows.append(dict(n_per_compartment=n, minutes_fine_grid=8 * (np.polyval(fit, 17235968) + n * np.polyval(per_decay, 17235968)) / 60, minutes_coarse_grid=8 * (np.polyval(fit, 4656960) + n * np.polyval(per_decay, 4656960)) / 60, days_training_split=one.sum() / 86400, days_training_and_test=one.sum() / 86400 * 227 / 180))
    cost = pd.DataFrame(rows)
    cost.to_csv(DATA / "cost_training_split.csv", index=False)
    print(cost.round(2).to_string(index=False))
    per_study_gb = {STUDY_NAME[s]: sum(f.stat().st_size for f in (DATA / s).glob("D_*_pet.nii.gz")) / 1e9 for s in R.STUDIES}
    print("GB per phantom, 8 PET-weighted D_c:", {k: round(v, 2) for k, v in per_study_gb.items()})
    print("VSV seconds per compartment:", vsv.groupby("study_id").vsv_seconds.median().rename(index=STUDY_NAME).round(1).to_dict())
    print("Unique grids in the split:", grids.to_dict())

**Proposal.**

**Monte Carlo for all phantoms, at about 5×10⁵ decays per compartment, not a subsample with VSV elsewhere.**

| | MC, 5×10⁵, 8 compartments | VSV/ρ |
|---|---|---|
| One phantom | 24 min fine grid, 14 min coarse (4 threads) | below 20 s |
| Training split (180) | **2.5 days** (3.2 with the 47 test studies) | about 1 h |
| Disk | about 4 GB in all (at most 30 MB per phantom) | none needed |
| Organ dose, statistics | SE below 1.8% | none |
| Organ dose, bias | none known: about 1% on self-dose, order 1 on cross-dose | -2 to -25% self, cross-dose missing, marrow down to -31% |

- **Why not a subsample:** the saving would be under 3 days, and VSV's errors depend on anatomy and on the TIA vector (bladder 1.35 on one phantom, 1.02 on the others), so a correction from a subsample would not transfer safely. 2×10⁵ (1.2 days) is enough for all but marrow and blood.
- **Voxel-level target: an open decision at this stage.** Per-voxel MC is noise at this budget, and 5×10⁶ would take 22 days. Three options, none tested here: (a) noise-free VSV/ρ rescaled per organ to the Monte Carlo mean, plus the smooth photon term of Monte Carlo; (b) train on the noisy MC map, whose noise is unbiased; (c) a kernel on the native PET grids, which removes the 4 mm blur, but not the missing long range.
- **Before the full run, to decide:** blood and spine as separate sources (10 runs instead of 8, about 3.1 days). Today marrow dose can only follow `rest_of_body`, while PBPK has its own marrow and blood TIA. If nb13 fits them separately, split now: rerunning later costs the whole budget.
- PET-weighted sources only: uniform adds nothing at organ level.

## Limitations

- **Three phantoms** are examples, not a population. The speed and cost come from two grid sizes.
- **The source is the 68Ga distribution at about 1 h** inside each compartment, used here as a spatial weight for 177Lu over days. It is a modelling choice of the virtual cohort, not tested here.
- **PET/CT misregistration and spill-out** stay in the sources, especially in `rest_of_body`. Activity outside the body mask has no source.
- **`rest_of_body` also contains blood pool, spine and kidney fluid,** so marrow and blood doses depend on it and cannot follow their own TIA.
- **Materials:** soft-tissue composition is assumed for the overridden organs. The parotids are at -48 to +60 HU, so the fixed 1.03 g/mL is sometimes above what the CT shows (fatty glands). GI tract and lesions keep the CT, including any contrast.
- **Statistics:** 3-5 batches, uniform sources one batch; the batch table holds plain means, so the SE of targets with gas (GI, rest of body) is approximate. Relative SEs are estimates, ±30-50%.
- **The test TIA vector is assumed,** so the shares of the dose and the total-dose errors are examples, not PBPK results.
- **OpenDose** is one reference anatomy, read at face value; the mapping of GI, bladder, red marrow and blood is ours. Cross-doses are not validated on measurements.
- Runs shared the machine with another session, so times can be somewhat pessimistic.